Setup some simple agents using Pydantic AI

In [1]:
from pydantic_ai import Agent
from dotenv import load_dotenv
load_dotenv('../.env')

# openai_api_key = os.environ.get("OPENAI_API_KEY")

True

A basic Pydantic AI run

In [12]:
async def ask_chatgpt():
    # Initialize the agent with Gemini model
    agent = Agent(
        'openai:gpt-5-mini',
        system_prompt='You are a helpful assistant specialized in Python programming.',
    )
    
    # Run the agent asynchronously
    result = await agent.run('Explain how to use structured outputs in Pydantic AI in a short response')
    print("ChatGPT Response:")
    print(result.output)

# Run the async function
await ask_chatgpt()

ChatGPT Response:
Structured outputs with Pydantic mean defining the schema you expect as Pydantic models, asking the LLM to return JSON that matches that schema, and then parsing/validating the LLM output with Pydantic.

Short recipe + example:

1. Define a model:
```python
from pydantic import BaseModel
from typing import List

class Product(BaseModel):
    name: str
    price: float
    tags: List[str] = []
```
2. Prompt the LLM to return JSON matching that schema (you can include Product.schema_json() in the prompt).
3. Parse/validate the LLM response:
```python
import json
# response_text is the LLM output (JSON)
product = Product.parse_raw(response_text)         # if it's a JSON string
# or
product = Product.parse_obj(json.loads(response_text))
```
4. Handle ValidationError to catch mismatches; you can coerce/fix or re-prompt the model.

Benefits: automatic type conversion, validation, nested models, default values, and clear error reporting. Use model.json_schema()/schema() to i

Async does not work well in a Jupyter Notebook. See stream_story.py for a version that works.

In [13]:
# Storyteller Agent
story_agent = Agent(
    'openai:gpt-5-mini',
    system_prompt="You are an AI storyteller. Generate engaging, real-time sci-fi adventures."
)

# Stream the story
async def stream_story():
    user_prompt = "Tell me a sci-fi story about a lost spaceship in a short response."
    async with story_agent.run_stream(user_prompt) as response:
        async for part in response.stream_text():
            print(part, end='', flush=True)

# Run the streaming story generator
await stream_story()

The ship wakes like a sleepwalker, systems coughing back to life under Captain Mira Solano's palms. Outside, map-sky has goneThe ship wakes like a sleepwalker, systems coughing back to life under Captain Mira Solano's palms. Outside, map-sky has gone blank — no star charts, no traffic lanes, only glassy dark and the slow drift of aThe ship wakes like a sleepwalker, systems coughing back to life under Captain Mira Solano's palms. Outside, map-sky has gone blank — no star charts, no traffic lanes, only glassy dark and the slow drift of a violet nebula that wasn't on any registry. Comms are dead; the crew manifest is a list that keeps refusing to load. The vessel itself smells of old coffee and ozone,The ship wakes like a sleepwalker, systems coughing back to life under Captain Mira Solano's palms. Outside, map-sky has gone blank — no star charts, no traffic lanes, only glassy dark and the slow drift of a violet nebula that wasn't on any registry. Comms are dead; the crew manifest is a li

Stateful agent test

In [11]:
# Initialize a stateful agent with the Gemini model
stateful_agent = Agent(
    'openai:gpt-5-mini',
    system_prompt='You are a conversational assistant that provides concise responses.',
)

# Initial user query
initial_query = 'Tell me about the Eiffel Tower in a short response.'

# Run the agent synchronously
initial_response = await stateful_agent.run(initial_query)
print(initial_response.output)
# Output: 'The Eiffel Tower is a wrought-iron lattice tower in Paris, France.'

# Follow-up query
follow_up_query = 'How tall is it? Answer without elaboration.'

# Run the agent with the follow-up query
follow_up_response = await stateful_agent.run(follow_up_query, message_history=initial_response.all_messages())
print('')
print(follow_up_response.output)
# Output: 'The Eiffel Tower is approximately 300 meters tall.'

follow_up_query2 = 'And how many times have con artists involved it in a scam?'
follow_up_response2 = await stateful_agent.run(follow_up_query2, message_history=follow_up_response.all_messages())
print('')
print(follow_up_response2.output)

The Eiffel Tower is a wrought-iron lattice tower in Paris, France, completed in 1889 for the Exposition Universelle and designed by Gustave Eiffel’s engineering company. It stands about 324 m (1,063 ft) tall with antennas, was built from 1887–1889, and is made of roughly 7,300 tons of iron. Initially controversial, it’s now an iconic symbol of France and one of the world’s most-visited paid monuments, drawing millions of visitors each year.

330 m (1,083 ft)

Impossible to know exactly. The most famous case: con artist Victor Lustig “sold” the Eiffel Tower twice (1925). Other scams/hoaxes have occurred, but there’s no definitive count.
